# CT4110 – Topic 10 Exercise
## From a symmetric QUBO to QAOA in Qiskit

This exercise develops a consistent mathematical formulation of number partitioning and carries it all the way into a working QAOA implementation.

You will learn to:

1. formulate number partitioning as
   $$
   f(\mathbf{x})=\mathbf{x}^{T}Q\mathbf{x}+\mathbf{v}^{T}\mathbf{x}+c,
   $$
   where $Q$ is symmetric;
2. absorb the linear term into the diagonal to obtain a symmetric matrix $Q'$;
3. convert the QUBO into the Ising cost Hamiltonian $H_C$;
4. implement $U_C(\gamma)=e^{-i\gamma H_C}$ and the mixer
   $U_M(\beta)=e^{-i\beta H_M}$ in Qiskit;
5. use a classical optimiser to choose QAOA angles;
6. compare QAOA with exact enumeration and a classical heuristic.

In [ ]:
# Run this cell first.

import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "qiskit": "qiskit>=2.2",
    "qiskit_optimization": "qiskit-optimization>=0.7",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from itertools import product

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.translators import to_ising

np.set_printoptions(precision=3, suppress=True)

## 1. Number partitioning as a symmetric QUBO

Let the positive weights be $w_0,\ldots,w_{n-1}$, and let

$$
W=\sum_{i=0}^{n-1}w_i.
$$

The binary variable $x_i$ records the group containing item $i$:

$$
x_i=
\begin{cases}
1,&\text{item }i\text{ is in group A},\\
0,&\text{item }i\text{ is in group B}.
\end{cases}
$$

The two group totals are

$$
S_A=\sum_i w_i x_i,
\qquad
S_B=W-S_A.
$$

We minimise the squared imbalance

$$
f(\mathbf{x})=(S_A-S_B)^2=(2\mathbf{w}^{T}\mathbf{x}-W)^2.
$$

Expanding gives

$$
f(\mathbf{x})
=\mathbf{x}^{T}Q\mathbf{x}+\mathbf{v}^{T}\mathbf{x}+c,
$$

with

$$
Q_{ij}=4w_iw_j,
\qquad
v_i=-4Ww_i,
\qquad
c=W^2.
$$

Because $x_i^2=x_i$ for binary variables, the linear term can be absorbed into the diagonal:

$$
Q'_{ij}=Q_{ij}+v_i\delta_{ij}
       =4\left(w_iw_j-Ww_i\delta_{ij}\right).
$$

Therefore,

$$
\boxed{f(\mathbf{x})=\mathbf{x}^{T}Q'\mathbf{x}+c}
$$

with $Q'$ symmetric.

Expanding this symmetric quadratic form once more gives

$$
\mathbf{x}^{T}Q'\mathbf{x}
=
\sum_i Q'_{ii}x_i
+
2\sum_{i<j}Q'_{ij}x_ix_j.
$$

The coefficient of the monomial $x_ix_j$ is therefore $2Q'_{ij}$, because the matrix entries $Q'_{ij}$ and $Q'_{ji}$ both contribute.

## 2. From the symmetric QUBO to an Ising Hamiltonian

Use the binary-to-spin substitution

$$
x_i=\frac{1-z_i}{2},
\qquad z_i\in\{+1,-1\}.
$$

For

$$
f(\mathbf{x})=\mathbf{x}^{T}Q'\mathbf{x}+c,
$$

the Ising energy is

$$
E(\mathbf{z})
=
\kappa+\sum_i h_i z_i+\sum_{i<j}J_{ij}z_iz_j.
$$

Substitution and collection of constant, linear, and quadratic spin terms gives

$$
\boxed{
h_i=-\frac12\sum_j Q'_{ij},
\qquad
J_{ij}=\frac12Q'_{ij}\quad(i<j),
\qquad
\kappa=c+\frac12\sum_iQ'_{ii}
              +\frac12\sum_{i<j}Q'_{ij}.
}
$$

For $h_i$, the index $j$ runs across every entry in row $i$. For $\kappa$,
the first sum uses the diagonal entries and the second uses each
off-diagonal pair once.

### Promote the Ising energy to an operator

Replace each spin value by a Pauli-$Z$ operator:

$$
z_i\longrightarrow Z_i.
$$

The cost Hamiltonian is

$$
\boxed{
H_C=\kappa I+\sum_i h_iZ_i+\sum_{i<j}J_{ij}Z_iZ_j.
}
$$

For every computational-basis state $|\mathbf{x}\rangle$,

$$
H_C|\mathbf{x}\rangle=f(\mathbf{x})|\mathbf{x}\rangle.
$$

Thus the basis states are eigenstates of $H_C$, and their eigenvalues are their QUBO scores.

### Number-partitioning simplification

For the absorbed matrix,

$$
\sum_jQ'_{ij}
=
4\left(w_i\sum_jw_j-Ww_i\right)=0.
$$

Consequently, every $h_i$ vanishes. Moreover,

$$
J_{ij}=2w_iw_j,
\qquad
\kappa=\sum_iw_i^2.
$$

Hence

$$
\boxed{
H_C=
\left(\sum_iw_i^2\right)I
+
2\sum_{i<j}w_iw_jZ_iZ_j.
}
$$

This result provides a direct check of the general conversion formulas.

## 3. Hamiltonians, unitaries, and the QAOA loop

A Hamiltonian is **Hermitian**; it is not generally unitary. The circuit uses its exponential:

$$
U_C(\gamma)=e^{-i\gamma H_C}.
$$

If

$$
H_C=\kappa I+\widetilde H_C,
$$

then

$$
U_C(\gamma)
=e^{-i\gamma\kappa}e^{-i\gamma\widetilde H_C}.
$$

The identity term contributes the global phase $e^{-i\gamma\kappa}$. It shifts every energy by $\kappa$ but cannot change measurement probabilities. We may therefore evolve with $\widetilde H_C$ and add $\kappa$ back when reporting the QUBO energy.

The standard mixer is

$$
H_M=\sum_i X_i,
\qquad
U_M(\beta)=e^{-i\beta H_M}
           =\bigotimes_i R_X(2\beta).
$$

A depth-$p$ QAOA state is

$$
|\psi_p(\boldsymbol\gamma,\boldsymbol\beta)\rangle
=
U_M(\beta_p)U_C(\gamma_p)\cdots
U_M(\beta_1)U_C(\gamma_1)
|+\rangle^{\otimes n}.
$$

Here

$$
\boldsymbol\gamma=(\gamma_1,\ldots,\gamma_p),
\qquad
\boldsymbol\beta=(\beta_1,\ldots,\beta_p).
$$

### Step 1: encode cost information as relative phase

Using the full Hamiltonian,

$$
U_C(\gamma)|\mathbf{x}\rangle
=
e^{-i\gamma f(\mathbf{x})}|\mathbf{x}\rangle.
$$

If the identity term is omitted from the circuit, the phase is instead
$e^{-i\gamma(f(\mathbf{x})-\kappa)}$. The two descriptions differ only by a common global phase, so all relative phases are unchanged.

### Step 2: convert relative phases into amplitudes

The mixer Hamiltonian $H_M$ connects computational-basis states at Hamming distance one. Its exponential $U_M(\beta)$ mixes amplitudes across this hypercube of bit strings. Depending on $\gamma$ and $\beta$, paths into a bit string can interfere constructively or destructively.

The mixer does **not** identify low-energy states by itself.

### Step 3: use classical feedback

The classical optimiser changes the angles to minimise

$$
F_p(\boldsymbol\gamma,\boldsymbol\beta)
=
\langle\psi_p(\boldsymbol\gamma,\boldsymbol\beta)|
H_C
|\psi_p(\boldsymbol\gamma,\boldsymbol\beta)\rangle
=
\sum_{\mathbf{x}}
P_{\boldsymbol\gamma,\boldsymbol\beta}(\mathbf{x})f(\mathbf{x}).
$$

Minimising this weighted average encourages probability to move toward lower-energy bit strings. It does not guarantee that every measurement is optimal.

QAOA is a structured variational algorithm because both alternating operators reflect the optimisation problem: $H_C$ contains the objective values and $H_M$ defines movement through the search space. This structure does not guarantee an advantage over a generic VQE ansatz or a classical method.

## 4. Implementing the two unitaries in Qiskit

Represent the non-identity part $\widetilde H_C$ with a `SparsePauliOp`. Then

```python
qc.append(PauliEvolutionGate(cost_operator, time=gamma), range(n))
```

implements

$$
e^{-i\gamma\widetilde H_C}.
$$

All cost terms in this exercise contain only $Z$ operators, so they commute.

The same cost unitary can be implemented term by term:

```python
qc.rz(2 * gamma * h_i, i)
qc.rzz(2 * gamma * J_ij, i, j)
```

because

$$
R_Z(\theta)=e^{-i\theta Z/2},
\qquad
R_{ZZ}(\theta)=e^{-i\theta Z\otimes Z/2}.
$$

Implement the mixer with one $X$ rotation per qubit:

```python
for qubit in range(n):
    qc.rx(2 * beta, qubit)
```

### Qiskit ordering

- Qubit $i$ represents variable $x_i$.
- Qiskit displays bit strings as $q_{n-1}\cdots q_0$.
- Reverse a displayed key before interpreting it as $(x_0,\ldots,x_{n-1})$.
- Pauli labels use the same right-to-left qubit ordering.

## 5. Math-to-code glossary

Only the non-obvious translations are listed below. A prefix such as `easy_`, `medium_`, or `hard_` identifies the task to which a variable belongs.

| Mathematics | Code name | Meaning |
|---|---|---|
| $Q'=Q+\operatorname{diag}(\mathbf{v})$ | `Qp` | Absorbed symmetric QUBO matrix. `Qp` means **Q prime**, not Q with a subscript $p$. |
| $h_i$ | `h[i]` | Coefficient of the $Z_i$ term in the Ising Hamiltonian. |
| $J_{ij}$ | `J[(i, j)]` | Coefficient of the $Z_iZ_j$ interaction. |
| $\kappa$ | `kappa` | Constant energy shift multiplying the identity. |
| $H_C$ | `cost_full` | Full cost Hamiltonian, including $\kappa I$. |
| $\widetilde H_C=H_C-\kappa I$ | `cost_evolution` | Identity-free Hamiltonian used inside the QAOA circuit. |
| Qiskit's returned $\widetilde H_C$ and $\kappa$ | `qiskit_operator`, `qiskit_offset` | The operator and constant returned by `to_ising`. |

For example, `easy_Qp` is $Q'$ for Task 1, while `medium_cost_full` is $H_C$ for Task 2.

---
# Task 1 — Easy: two-variable partition

Begin with the two weights

$$
\mathbf{w}=(5,9),
\qquad W=14.
$$

For this problem,

$$
Q=
\begin{pmatrix}
100&180\\
180&324
\end{pmatrix},
\qquad
\mathbf{v}=
\begin{pmatrix}
-280\\-504
\end{pmatrix},
\qquad
c=196.
$$

Your first QUBO step is to calculate the absorbed matrix yourself using

$$
Q'=Q+\operatorname{diag}(\mathbf{v}).
$$

Do not enter $Q'$ directly: construct it from $Q$ and $\mathbf{v}$ in Task 1.2. You will then verify the four binary assignments, derive $H_C$, and implement one QAOA layer in two equivalent ways.

In [ ]:
# Task 1.1 — classical reference level
#
# Instructions
# 1. Define partition_score(bits, weights).
# 2. Return (S_A, S_B, imbalance, squared_imbalance), where S_A and S_B
#    are the two group totals from the mathematical formulation.
# 3. Enumerate 00, 01, 10, and 11.
# 4. Store the records in easy_results and print them.
# 5. Identify every assignment with the minimum squared imbalance.

easy_weights = np.array([5, 9], dtype=float)

def partition_score(bits, weights):
    # YOUR CODE HERE
    raise NotImplementedError

easy_results = []
# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print every record in easy_results, showing bits, S_A, S_B, imbalance,
#   and squared_imbalance.
# - Print the minimum squared_imbalance in easy_results.
# - Print every bits value attaining that minimum.


In [ ]:
# Task 1.2 — verify the symmetric QUBO
#
# Instructions
# 1. Construct easy_Q (Q), easy_v (v), and easy_c (c) from the formulas above.
# 2. Calculate easy_Qp (Q prime) using
#       easy_Qp = easy_Q + np.diag(easy_v)
#    Do not hard-code easy_Qp.
# 3. Check that easy_Q and easy_Qp are symmetric.
# 4. Define qubo_score(bits, Qp, constant) so it returns
#       bits.T @ Qp @ bits + constant.
# 5. For all four bit strings, compare qubo_score(bits, easy_Qp, easy_c)
#    with the squared imbalance returned by partition_score.
# 6. Calculate and print one Boolean check showing whether all four pairs agree.

easy_Q = None
easy_v = None
easy_Qp = None
easy_c = None

def qubo_score(bits, Qp, constant):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print easy_Q, easy_v, easy_Qp, and easy_c.
# - Print the symmetry checks for easy_Q and easy_Qp.
# - For all four bits values, print the squared imbalance returned by
#   partition_score(bits, easy_weights) and the value returned by
#   qubo_score(bits, easy_Qp, easy_c).
# - Print one final Boolean check showing whether all four pairs agree.


In [ ]:
# Task 1.3 — derive and build the cost Hamiltonian
#
# Calculate the code-level quantities from easy_Qp (Q prime):
#   easy_h[i]      = -0.5 * sum_j easy_Qp[i, j]
#   easy_J[(i, j)] =  0.5 * easy_Qp[i, j]       for i < j
#   easy_kappa     =  easy_c + 0.5 * sum_i easy_Qp[i, i]
#                     + 0.5 * sum_{i<j} easy_Qp[i, j]
#
# Instructions
# 1. Calculate easy_h (h), easy_J (J), and easy_kappa (kappa).
# 2. Check easy_h against [0, 0], easy_J[(0, 1)] against 90, and
#    easy_kappa against 106.
# 3. Build easy_cost_evolution (H_tilde = 90 Z_0 Z_1) as a SparsePauliOp.
#    For two qubits the Pauli label is "ZZ".
# 4. Build easy_cost_full (H_C = 106 I + H_tilde).
# 5. For every bits value, compare
#       Statevector.from_label(...).expectation_value(easy_cost_full)
#    with qubo_score(bits, easy_Qp, easy_c).
#
# Hint:
#   Statevector.from_label("01").expectation_value(operator)
# uses the displayed order |q_1 q_0>. Convert carefully from (x_0, x_1).

easy_h = None
easy_J = {}
easy_kappa = None
easy_cost_evolution = None
easy_cost_full = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print easy_h, easy_J, and easy_kappa.
# - Print easy_cost_evolution and easy_cost_full.
# - Print one row per bits value showing qubo_score(bits, easy_Qp, easy_c)
#   and the corresponding easy_cost_full eigenvalue.
# - Print one final Boolean check showing whether all four easy_cost_full
#   eigenvalues match qubo_score(bits, easy_Qp, easy_c).


### Qiskit Optimization: representing and checking the QUBO

`qiskit_optimization` provides classes for writing optimisation problems in a standard form. Its `QuadraticProgram` class stores the variables, objective function, and any constraints. Here it gives us an independent representation of the same unconstrained binary objective already stored in `easy_Qp` and `easy_c`.

Create the model and add its binary variables in the same order as the qubits:

```python
easy_quad = QuadraticProgram("two_item_partition")
easy_quad.binary_var("x0")
easy_quad.binary_var("x1")
```

Set the objective $\mathbf{x}^{T}Q'\mathbf{x}+c$. The separate linear argument is unnecessary because it has already been absorbed into `easy_Qp`:

```python
easy_quad.minimize(constant=easy_c, quadratic=easy_Qp)
print(easy_quad.prettyprint())
```

Evaluate the stored objective at a candidate `bits` value and compare it with the direct implementation:

```python
qiskit_value = easy_quad.objective.evaluate(bits)
direct_value = qubo_score(bits, easy_Qp, easy_c)
```

Finally, translate the binary objective into an Ising operator:

```python
easy_qiskit_operator, easy_qiskit_offset = to_ising(easy_quad)
```

`easy_qiskit_operator` is the identity-free operator $\widetilde H_C$, while `easy_qiskit_offset` is the constant $\kappa$. The full Hamiltonian is therefore

$$
H_C=\widetilde H_C+\kappa I.
$$

Relevant official documentation:

- [Quadratic Programs tutorial](https://qiskit-community.github.io/qiskit-optimization/tutorials/01_quadratic_program.html)
- [`QuadraticProgram` API](https://qiskit-community.github.io/qiskit-optimization/stubs/qiskit_optimization.QuadraticProgram.html)
- [`to_ising` API](https://qiskit-community.github.io/qiskit-optimization/stubs/qiskit_optimization.translators.to_ising.html)

In [ ]:
# Task 1.3b — represent the same QUBO with Qiskit Optimization
#
# Instructions
# 1. Set easy_quad = QuadraticProgram("two_item_partition").
# 2. Add x0 and x1 to easy_quad as binary variables, in that order.
# 3. Call easy_quad.minimize(constant=easy_c, quadratic=easy_Qp). The full
#    symmetric array preserves the expression x.T @ Qp @ x.
# 4. For every bits value, compare easy_quad.objective.evaluate(bits) with
#    qubo_score(bits, easy_Qp, easy_c).
# 5. Assign to_ising(easy_quad) to
#    (easy_qiskit_operator, easy_qiskit_offset).
# 6. Check easy_qiskit_offset against easy_kappa and check that
#    easy_qiskit_operator + easy_qiskit_offset*I equals easy_cost_full.
#
# This checkpoint connects the mathematical notation, Qiskit's optimisation model,
# and the SparsePauliOp that is evolved by the QAOA circuit.

easy_quad = None
easy_qiskit_operator = None
easy_qiskit_offset = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print easy_quad using prettyprint().
# - For all four bits values, print easy_quad.objective.evaluate(bits) and
#   qubo_score(bits, easy_Qp, easy_c).
# - Print easy_qiskit_operator and easy_qiskit_offset.
# - Print whether easy_qiskit_offset equals easy_kappa.
# - Print whether easy_qiskit_operator + easy_qiskit_offset*I equals
#   easy_cost_full.


In [ ]:
# Task 1.4 — implement U_C and U_M in two equivalent ways
#
# Use gamma = 0.31 and beta = -0.22 for this circuit-equivalence check.
#
# Circuit A:
#   1. Apply H to both qubits to prepare |++>.
#   2. Append PauliEvolutionGate(easy_cost_evolution, time=gamma).
#   3. Apply RX(2*beta) to both qubits.
#
# Circuit B:
#   1. Apply H to both qubits.
#   2. Apply RZZ(2*gamma*easy_J[(0, 1)]) to qubits (0, 1).
#   3. Apply RX(2*beta) to both qubits.
#
# Compare Statevector.from_instruction(easy_qaoa_evolution) with
# Statevector.from_instruction(easy_qaoa_direct) using .equiv(...).
# They should agree up to global phase.
# Draw one circuit with output="mpl".

gamma = 0.31
beta = -0.22

easy_qaoa_evolution = QuantumCircuit(2)
easy_qaoa_direct = QuantumCircuit(2)

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Draw easy_qaoa_evolution using output="mpl".
# - Draw easy_qaoa_direct using output="mpl".
# - Print whether the two final Statevectors are equivalent up to global phase;
#   the result must be True.


In [ ]:
# Task 1.5 — let a classical optimiser choose the p=1 angles
#
# Instructions
# 1. Define build_qaoa(cost_evolution, gammas, betas).
#    - infer n from cost_evolution.num_qubits
#    - require len(gammas) == len(betas)
#    - prepare |+>^n
#    - alternate PauliEvolutionGate(cost_evolution, time=gamma_l)
#      and RX(2*beta_l) on every qubit
# 2. Define qaoa_energy(parameters, cost_evolution, cost_full, p).
#    - first p entries are gammas; the remaining p entries are betas
#    - construct the unmeasured circuit
#    - use Statevector.from_instruction(circuit)
#    - return the real expectation value of cost_full
# 3. Set easy_optimisation using scipy.optimize.minimize with
#    easy_initial, method="COBYLA", and p=1.
# 4. Compare easy_optimisation.fun with the exact minimum, 16.
#
# Note: the identity term is absent from the evolution circuit but present in
# cost_full, so the reported objective remains on the original QUBO scale.

def build_qaoa(cost_evolution, gammas, betas):
    # YOUR CODE HERE
    raise NotImplementedError

def qaoa_energy(parameters, cost_evolution, cost_full, p):
    # YOUR CODE HERE
    raise NotImplementedError

easy_initial = np.array([0.2, -0.2])
easy_optimisation = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print easy_initial and qaoa_energy(easy_initial, easy_cost_evolution,
#   easy_cost_full, p=1).
# - Print easy_optimisation.x, easy_optimisation.fun, easy_optimisation.success,
#   and easy_optimisation.message.
# - Print abs(easy_optimisation.fun - 16).


In [ ]:
# Task 1.6 — inspect the final probability distribution
#
# Instructions
# 1. Build the p=1 circuit from easy_optimisation.x.
# 2. Store its Statevector in easy_final_state.
# 3. Set easy_probabilities = easy_final_state.probabilities_dict().
# 4. Plot easy_probabilities with plot_histogram.
# 5. Reverse each displayed key before treating it as (x_0, x_1).
# 6. Store the total probability of the optimal assignments in
#    easy_optimal_probability.
#
# The optimiser minimises an expectation, not a single measured bit string.
# A final run must therefore be sampled and the low-cost strings identified.

easy_final_state = None
easy_probabilities = None
easy_optimal_probability = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print all entries in easy_probabilities in descending probability order.
# - Print easy_optimal_probability.
# - Display plot_histogram(easy_probabilities).


---
# Task 2 — Medium: four-variable vehicle load balancing

A depot must divide four indivisible loads between two vehicles:

$$
\mathbf{w}=(8,7,6,5).
$$

The binary variable $x_i=1$ assigns load $i$ to vehicle A; $x_i=0$ assigns it to vehicle B.

You will now generalise every step:

1. obtain the exact classical reference;
2. construct the symmetric $Q'$;
3. convert any symmetric $Q'$ into a Qiskit cost operator;
4. create and optimise a reusable QAOA circuit;
5. decode the most probable assignments and compare them with the exact answer.

In [ ]:
# Task 2.1 — exact classical reference and symmetric QUBO
#
# Instructions
# 1. Store medium_weights = [8, 7, 6, 5].
# 2. Store all 16 partition_score results in medium_exact_results and every
#    optimal bits value in medium_optimal_bits.
# 3. Define partition_qubo(weights), returning (Q, v, Qp, c) using:
#       Q  = 4 * outer(w, w)
#       v  = -4 * W * w
#       Qp = Q + diag(v)
#       c  = W**2
#    Then set medium_Q, medium_v, medium_Qp, medium_c using
#    partition_qubo(medium_weights).
# 4. Check that medium_Qp (Q prime) is symmetric and every row sums to zero.
# 5. For all 16 bits values, compare
#       qubo_score(bits, medium_Qp, medium_c)
#    with the squared imbalance returned by partition_score(bits, medium_weights).

medium_weights = np.array([8, 7, 6, 5], dtype=float)

def partition_qubo(weights):
    # YOUR CODE HERE
    raise NotImplementedError

medium_exact_results = []
medium_optimal_bits = []
medium_Q = medium_v = medium_Qp = medium_c = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print all records in medium_exact_results, including bits, S_A, S_B,
#   and squared_imbalance.
# - Print the minimum squared_imbalance and medium_optimal_bits.
# - Print medium_Qp and medium_c.
# - Print the medium_Qp symmetry check, medium_Qp row sums, and one final
#   Boolean check for the 16 partition_score/qubo_score comparisons.


In [ ]:
# Task 2.2 — generic symmetric-QUBO to Ising conversion
#
# Complete symmetric_qubo_to_ising(Qp, constant).
#
# It must return:
#   cost_full       = kappa*I + sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
#   cost_evolution  = the same operator without kappa*I
#   kappa, h, J
#
# Qiskit Pauli labels are ordered q_(n-1)...q_0. To place Z on qubit i:
#   label = ["I"] * n
#   label[n - 1 - i] = "Z"
#
# Skip coefficients whose absolute value is below atol. Finish each operator
# with .simplify().
#
# Verification:
# 1. Set medium_cost_full, medium_cost_evolution, medium_kappa, medium_h,
#    and medium_J using symmetric_qubo_to_ising(medium_Qp, medium_c).
# 2. Check that medium_h (h) is zero for this partitioning problem.
# 3. For every bits value, compare the medium_cost_full eigenvalue with
#    qubo_score(bits, medium_Qp, medium_c).
# 4. Compare medium_kappa (kappa) with np.sum(medium_weights**2).

def symmetric_qubo_to_ising(Qp, constant, atol=1e-12):
    # YOUR CODE HERE
    raise NotImplementedError

medium_cost_full = None
medium_cost_evolution = None
medium_kappa = None
medium_h = None
medium_J = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print medium_kappa, medium_h, and every entry of medium_J.
# - Print medium_cost_evolution and medium_cost_full.
# - Print the maximum absolute difference between
#   qubo_score(bits, medium_Qp, medium_c) and the corresponding
#   medium_cost_full eigenvalue.
# - Print whether medium_h is zero and whether medium_kappa equals
#   sum(medium_weights**2).


In [ ]:
# Task 2.3 — optimise a p=1 QAOA circuit
#
# Instructions
# 1. Reuse build_qaoa and qaoa_energy from Task 1 with
#    medium_cost_evolution and medium_cost_full.
# 2. Run COBYLA once from each entry in medium_starts. Several starts are
#    used because the variational objective may have local minima.
# 3. Append every optimiser result to medium_runs.
# 4. Set medium_best to the entry in medium_runs with the smallest .fun.
# 5. Compare medium_best.fun with the exact minimum in medium_exact_results.
# 6. Draw build_qaoa(...) using the angles in medium_best.x.
#
# Suggested starts are supplied, but you may add more.

medium_starts = [
    np.array([0.1, 0.1]),
    np.array([0.5, -0.3]),
    np.array([1.0, 0.4]),
    np.array([1.5, -0.7]),
]

medium_runs = []
medium_best = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - For every entry in medium_runs, print its starting point, .fun, .success,
#   and .nfev.
# - Print medium_best.x, medium_best.fun, the exact minimum from
#   medium_exact_results, and their absolute difference.
# - Draw the final p=1 QAOA circuit using output="mpl".


In [ ]:
# Task 2.4 — decode probabilities and evaluate solution quality
#
# Instructions
# 1. Build the circuit at medium_best.x and store its Statevector in
#    medium_final_state.
# 2. Set medium_probabilities = medium_final_state.probabilities_dict(), then
#    sort its items from most to least probable.
# 3. For each of the eight most probable keys:
#       bits = np.array([int(b) for b in key[::-1]])
#    Then report probability, bits, partition_score(bits, medium_weights),
#    and qubo_score(bits, medium_Qp, medium_c).
# 4. Store the probability of sampling an entry in medium_optimal_bits in
#    medium_optimal_probability.
# 5. Reconstruct the expected score from medium_probabilities and compare it
#    with medium_best.fun.
# 6. Plot medium_probabilities.

medium_final_state = None
medium_probabilities = None
medium_optimal_probability = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the eight largest entries in medium_probabilities with the displayed
#   Qiskit key, decoded bits, probability, partition_score(bits, medium_weights),
#   and qubo_score(bits, medium_Qp, medium_c).
# - Print medium_optimal_probability.
# - Print the expected score reconstructed from medium_probabilities.
# - Print its absolute difference from medium_best.fun.
# - Display plot_histogram(medium_probabilities).


---
# Advanced Task 3 — Hard: six variables, depth-2 QAOA, and a heuristic solver

Two vans must share six indivisible containers:

$$
\mathbf{w}=(3,5,6,7,9,11).
$$

The total is odd, so a zero imbalance is impossible. Your goals are to:

1. build the six-variable symmetric QUBO and exact benchmark;
2. implement and optimise depth-$2$ QAOA;
3. implement a one-bit-flip hill climber with random restarts *(implementation details are provided at the corresponding checkpoints)*;
4. compare exact search, depth-$2$ QAOA, and the heuristic.

At depth $p=2$, the QAOA circuit is

$$
U_M(\beta_2)U_C(\gamma_2)
U_M(\beta_1)U_C(\gamma_1)|+\rangle^{\otimes 6}.
$$

It contains four variational parameters:
$\gamma_1,\gamma_2,\beta_1,$ and $\beta_2$.

### Random-restart hill climbing

A single hill-climbing run begins from one bit string and repeatedly moves to
the best lower-cost one-bit neighbour. It stops when none of those neighbours
improves the score, so it may finish at a local rather than global minimum.

Random-restart hill climbing repeats this procedure from several independently
generated starting bit strings and retains the best final result. Increasing
the restart budget explores more regions of the search space, but also requires
more classical objective evaluations. It improves the chance of finding the
global optimum without guaranteeing it.

In [ ]:
# Task 3.1 — exact benchmark, QUBO, and Hamiltonian
#
# Instructions
# 1. Store all 64 partition_score results in hard_exact_results and every
#    optimal bits value in hard_optimal_bits.
# 2. Set hard_Q, hard_v, hard_Qp, and hard_c using
#    partition_qubo(hard_weights).
# 3. Set hard_cost_full, hard_cost_evolution, hard_kappa, hard_h, and hard_J
#    using symmetric_qubo_to_ising(hard_Qp, hard_c).
# 4. For all 64 bits values, compare the hard_cost_full eigenvalue with both
#    partition_score(bits, hard_weights) and qubo_score(bits, hard_Qp, hard_c).
# 5. Print len(hard_cost_evolution) to report the number of non-identity
#    Pauli terms.

hard_weights = np.array([3, 5, 6, 7, 9, 11], dtype=float)

hard_exact_results = []
hard_optimal_bits = []
hard_Q = hard_v = hard_Qp = hard_c = None
hard_cost_full = hard_cost_evolution = None
hard_kappa = hard_h = hard_J = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the minimum squared_imbalance in hard_exact_results,
#   len(hard_optimal_bits), and hard_optimal_bits.
# - Print hard_Qp, hard_c, hard_kappa, and hard_h.
# - Print the number of Pauli terms in hard_cost_evolution.
# - Print the maximum absolute difference between
#   qubo_score(bits, hard_Qp, hard_c) and the corresponding hard_cost_full
#   eigenvalue.


In [ ]:
# Task 3.2 — optimise depth-2 QAOA
#
# Instructions
# 1. Use rng to generate at least eight reproducible starting vectors in the
#    order [gamma_1, gamma_2, beta_1, beta_2].
# 2. Run COBYLA from every starting vector using qaoa_energy with
#    hard_cost_evolution, hard_cost_full, and p=2.
# 3. Append every optimiser result to hard_p2_runs and set hard_p2_best to
#    the entry with the smallest .fun.
# 4. Using hard_p2_best, calculate:
#       - best expected QUBO score;
#       - probability of sampling an exact optimum;
#       - the most probable decoded bit string and its score;
#       - number of objective evaluations.
# 5. Store the final depth-2 distribution in hard_p2_probabilities.
#
# Suggested optimiser settings:
#   method="COBYLA", options={"maxiter": 250}

rng = np.random.default_rng(4110)
hard_p2_runs = []
hard_p2_best = None
hard_p2_probabilities = None

# YOUR CODE HERE

# REQUIRED OUTPUT
# - For every entry in hard_p2_runs, print its starting vector, .fun, .success,
#   and .nfev.
# - Print hard_p2_best.x, hard_p2_best.fun, the exact minimum from
#   hard_exact_results, and their absolute difference.
# - Print the exact-optimum probability and the most probable decoded bit
#   string with its QUBO score.
# - Display plot_histogram(hard_p2_probabilities).


In [ ]:
# Task 3.3 — implement one-bit-flip hill climbing
#
# Instructions
# 1. Define one_bit_neighbours(bits). For n bits it must return n neighbours,
#    each differing in exactly one position.
# 2. Define hill_climb(start_bits, Qp, constant), using
#    qubo_score(neighbour, Qp, constant) to score each neighbour.
# 3. At each step, move to the neighbour with the strictly lowest score.
# 4. Stop if no neighbour has a lower score.
# 5. Return the final bits, final score, and the full history.
# 6. Test hill_climb(start_bits, hard_Qp, hard_c) from at least three chosen
#    starting bit strings.
#
# Use qubo_score as the objective so the heuristic and QAOA solve exactly the
# same mathematical problem.

def one_bit_neighbours(bits):
    # YOUR CODE HERE
    raise NotImplementedError

def hill_climb(start_bits, Qp, constant):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

# REQUIRED OUTPUT
# - For one start_bits value, print one_bit_neighbours(start_bits) and a
#   Boolean check that every returned value has Hamming distance one.
# - For at least three start_bits values, print the values returned by
#   hill_climb(start_bits, hard_Qp, hard_c), including the final bits, final
#   score, number of moves, and complete score history.


In [ ]:
# Task 3.4 — random-restart heuristic
#
# Instructions
# 1. Define random_restart_hill_climb(Qp, constant, restarts, seed).
# 2. Inside the function, generate each start_bits value using
#    np.random.default_rng(seed).
# 3. Run hill_climb(start_bits, Qp, constant) from every start_bits value.
# 4. Return the best result and all individual run records.
# 5. Iterate over restart_budgets = [1, 5, 20, 100].
# 6. For each budget, call
#       random_restart_hill_climb(hard_Qp, hard_c, budget, seed)
#    for 20 different seed values.
# 7. Report:
#       - exact-optimum success rate;
#       - mean best score;
#       - worst best score;
#       - mean number of hill-climb moves.
#    Store these summaries in heuristic_summary.

def random_restart_hill_climb(Qp, constant, restarts, seed):
    # YOUR CODE HERE
    raise NotImplementedError

restart_budgets = [1, 5, 20, 100]
heuristic_summary = {}

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print every entry in heuristic_summary, including its exact-optimum
#   success rate, mean best score, worst best score, and mean move count.
# - Print the best bits value and score returned for restart_budgets[-1].


In [ ]:
# Task 3.5 — final comparison
#
# Build a concise comparison table with one row for each method:
#   exact enumeration, depth-2 QAOA, and 100-restart hill climbing.
#
# Suggested columns:
#   method
#   best bit-string score found
#   expected score (QAOA only)
#   exact-optimum sampling/success probability
#   main computational cost
#
# Make clear that:
# - a QAOA expectation is not itself a sampled solution;
# - statevector simulation is a classical calculation used here for learning;
# - the hill-climber's success rate is empirical;
# - exact enumeration is feasible here only because n=6.

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Display one comparison table with exactly three rows: exact enumeration,
#   depth-2 QAOA, and 100-restart hill climbing.
# - Include the best sampled/found score, expected score where applicable,
#   exact-optimum probability or success rate, and main computational cost.
